In [ ]:
# Embedding 방법 정리
# ChromaDB는 임베딩 모델이 아니라 임베딩 결과를 저장하는 DB
# 임베딩이 선행 - 방법이 여러가지

!pip install chromadb sentence-transformers

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path='.chroma_db')
texts = ['사과는 과일이야', '고양이는 동물이야']

In [ ]:
print('방법1 : 이런 것도 가능하나 비권장')
# ChromaDB 내부 임베딩 함수를 직접 꺼내 사용하는 방식
collection1 = client.get_or_create_collection(name="test")
embedding_fn1 = collection1._embedding_function
embeddings1 = embedding_fn1(texts)
print(len(embeddings1), len(embeddings1[0]))
print(embeddings1[0][:5])

collection1.upsert(  # update + insert
    documents=texts,
    embeddings = embeddings1,
    ids=["id1", "id2"]
)


In [ ]:
print('방법2 : ChromaDB에서 가장 일반적인 방법')
# ChromaDB에 임베딩 함수를 등록해 자동 임베딩하는 방식
from chromadb.utils import embedding_functions

embedding_fn2 = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
embeddings2 = embedding_fn2(texts)
print(len(embeddings2), len(embeddings2[0]))
print(embeddings2[0][:5])

collection2 = client.get_or_create_collection(name="test2", embedding_function=embedding_fn2)
collection2.upsert(
    documents=texts,
    ids=["id1", "id2"]
)


In [ ]:
print('방법3 : ChromaDB와 별개로 SentenceTransformer 모델을 직접 사용해 임베딩을 만드는 방법')
# SentenceTransformer로 직접 임베딩
model3 = SentenceTransformer("all-MiniLM-L6-v2")
embeddings3 = model3.encode(texts).tolist()
print(len(embeddings3), len(embeddings3[0]))
print(embeddings3[0][:5])

collection3 = client.get_or_create_collection(name="test3")
collection3.upsert(
    documents=texts,
    embeddings = embeddings3,
    ids=["id1", "id2"]
)


In [ ]:
print('방법4 : Hugging Face의 사전학습 모델을 로컬에서 방법')
embedding_fn4 = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='jhgan/ko-sroberta-multitask'
)
embeddings4 = embedding_fn4(texts)
print(len(embeddings4), len(embeddings4[0]))
print(embeddings4[0][:5])

collection4 = client.get_or_create_collection(name="test4", embedding_function=embedding_fn4)
collection4.upsert(
    documents=texts,
    ids=["id1", "id2"]
)
